# 02 Clean

This notebook completes the required cleaning workflow for P01. It reads raw CSV files from `data/stock/`, `data/index/`, and `data/combined/combined_macro_indicators.csv`, then produces cleaned CSV and Parquet files plus merged datasets for later analysis.

Main outputs:
- `data/clean/stock_clean.csv`
- `data/clean/stock_clean.parquet`
- `data/clean/stock_close_wide.csv`
- `data/clean/stock_close_long.csv`
- `data/combined/stock_index_daily.csv`
- `data/combined/combined_data.csv`


## 0. Setup

Before cleaning, the notebook imports the required packages and creates output folders if they do not already exist. This keeps all generated files inside the project structure required by the assignment.


In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATA_DIR = Path("data")
STOCK_DIR = DATA_DIR / "stock"
INDEX_DIR = DATA_DIR / "index"
MACRO_DIR = DATA_DIR / "macro"
CLEAN_DIR = DATA_DIR / "clean"
COMBINED_DIR = DATA_DIR / "combined"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_DIR.mkdir(parents=True, exist_ok=True)


## 3.1 Single-table Cleaning

Each raw stock file is cleaned independently. Before cleaning, the raw files may contain inconsistent data types, duplicated trading dates, or missing values caused by suspension, holidays, or source-side gaps. After cleaning, all tables share the same column names, `date` is converted to `datetime64[ns]` and used as the index, price and trading columns are numeric, duplicated `date + code` rows are removed, missing numeric values are handled, and extreme daily returns are labelled.


In [2]:
stock_columns = ["date", "code", "name", "industry", "open", "close", "high", "low", "volume", "amount"]
numeric_cols = ["open", "close", "high", "low", "volume", "amount"]
key_cols = ["date", "code"]

missing_tables = []
clean_frames = []
clean_log = []
type_log = []

for path in sorted(STOCK_DIR.glob("stock_*.csv")):
    raw = pd.read_csv(path, dtype=str)
    raw_rows = len(raw)
    code = path.stem.replace("stock_", "")

    # Standardize by position because the source files use Chinese headers.
    # This avoids encoding-dependent failures when the notebook is executed on Windows.
    df = raw.copy()
    df.columns = stock_columns[:len(df.columns)]

    missing = df.isna().sum().reset_index()
    missing.columns = ["column", "missing_count"]
    missing["missing_ratio"] = np.where(raw_rows == 0, np.nan, missing["missing_count"] / raw_rows)
    missing["code"] = code
    missing_tables.append(missing[["code", "column", "missing_count", "missing_ratio"]])

    df["code"] = df["code"].astype(str).str.extract(r"(\d{6})", expand=False).fillna(code)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    converted_cols = []
    for col in numeric_cols:
        before_dtype = df[col].dtype
        df[col] = pd.to_numeric(df[col].str.replace(",", "", regex=False), errors="coerce")
        after_dtype = df[col].dtype
        if str(before_dtype) != str(after_dtype):
            converted_cols.append(f"{col}: {before_dtype} -> {after_dtype}")

    duplicate_count = int(df.duplicated(subset=key_cols).sum())
    df = df.drop_duplicates(subset=key_cols, keep="last")

    missing_before_fill = int(df[numeric_cols].isna().sum().sum())
    df = df.sort_values("date").set_index("date")

    # Use forward fill for numeric fields because prices and trading fields are time-series data.
    # Rows that still lack essential numeric data after ffill are removed.
    df[numeric_cols] = df[numeric_cols].ffill()
    rows_before_dropna = len(df)
    df = df.dropna(subset=numeric_cols)
    rows_dropped_after_fill = int(rows_before_dropna - len(df))

    df["return"] = df["close"].pct_change()
    df["is_extreme"] = df["return"].abs() > 0.20
    df["year"] = df.index.year.astype("int16")

    type_log.append({
        "code": code,
        "converted_columns": "; ".join(converted_cols) if converted_cols else "none",
    })
    clean_log.append({
        "code": code,
        "raw_rows": raw_rows,
        "duplicate_rows_deleted": duplicate_count,
        "missing_values_before_fill_numeric": missing_before_fill,
        "rows_dropped_after_ffill": rows_dropped_after_fill,
        "clean_rows": len(df),
        "extreme_rows": int(df["is_extreme"].sum()),
    })
    clean_frames.append(df.reset_index())

missing_summary = pd.concat(missing_tables, ignore_index=True)
clean_log_df = pd.DataFrame(clean_log)
type_log_df = pd.DataFrame(type_log)
stock_clean = pd.concat(clean_frames, ignore_index=True).sort_values(["code", "date"])
stock_clean = stock_clean.set_index("date").sort_index()

print("Missing-value summary by stock and column")
display(missing_summary)
print("Cleaning log")
display(clean_log_df)
print("Data type conversion log")
display(type_log_df)
print(stock_clean.dtypes)


Missing-value summary by stock and column


,code,column,missing_count,missing_ratio
0,000002,date,0,0.0
1,000002,code,0,0.0
2,000002,name,0,0.0
3,000002,industry,0,0.0
4,000002,open,0,0.0
...,...,...,...,...
95,601398,close,0,0.0
96,601398,high,0,0.0
97,601398,low,0,0.0
98,601398,volume,0,0.0


Cleaning log


,code,raw_rows,duplicate_rows_deleted,missing_values_before_fill_numeric,rows_dropped_after_ffill,clean_rows,extreme_rows
0,000002,1540,0,0,0,1540,0
1,000858,1540,0,0,0,1540,0
2,002352,1537,0,0,0,1537,0
3,002594,1540,0,0,0,1540,0
4,300308,1540,0,0,0,1540,2
5,600036,1540,0,0,0,1540,0
6,600519,1540,0,0,0,1540,0
7,601088,1530,0,0,0,1530,0
8,601127,1539,0,0,0,1539,0
9,601398,1540,0,0,0,1540,0


Data type conversion log


,code,converted_columns
0,000002,open: object -> float64; close: object -> floa...
1,000858,open: object -> float64; close: object -> floa...
2,002352,open: object -> float64; close: object -> floa...
3,002594,open: object -> float64; close: object -> floa...
4,300308,open: object -> float64; close: object -> floa...
5,600036,open: object -> float64; close: object -> floa...
6,600519,open: object -> float64; close: object -> floa...
7,601088,open: object -> float64; close: object -> floa...
8,601127,open: object -> float64; close: object -> floa...
9,601398,open: object -> float64; close: object -> floa...


code           object
name           object
industry       object
open          float64
close         float64
high          float64
low           float64
volume        float64
amount        float64
return        float64
is_extreme       bool
year            int16
dtype: object


### Explanation of 3.1 Changes

Missing-value detection is performed before any filling, so the table records the original data quality by stock and column. Possible reasons include stock trading suspension, non-trading days, and temporary data-source gaps. Because the input files are already trading-day observations, ordinary weekends and holidays usually appear as missing dates rather than missing rows.

Missing numeric values are forward-filled because daily stock prices are time-series observations and the most recent valid quote is usually the least disruptive replacement for short gaps. Rows that still lack key numeric values after forward fill are deleted because they cannot support return calculation or regression analysis. Duplicate rows are removed on `date + code`; the log records how many rows were deleted. Extreme observations are labelled, not deleted, because daily moves above +/-20% may reflect limit-up/limit-down rules, corporate events, suspensions followed by resumed trading, or data errors that should be inspected later.


## Save Clean Stock Data as CSV and Parquet

After single-table cleaning, the combined stock panel is saved as the required CSV file. A Parquet copy is also written to demonstrate the advanced storage format selected for this project.


In [3]:
stock_clean_out = stock_clean.reset_index()
stock_clean_out.to_csv(CLEAN_DIR / "stock_clean.csv", index=False, encoding="utf-8-sig")
stock_clean_out.to_parquet(CLEAN_DIR / "stock_clean.parquet", index=False)

print("stock_clean.csv", stock_clean_out.shape)
print("stock_clean.parquet", stock_clean_out.shape)


stock_clean.csv (15386, 13)
stock_clean.parquet (15386, 13)


## Parquet Demonstration

Parquet supports columnar reads and a clear schema. The following cells load only `date`, `code`, and `close`, inspect the Parquet schema, and compare read time and file size with CSV.


In [4]:
df_parquet_cols = pd.read_parquet(
    "data/clean/stock_clean.parquet",
    columns=["date", "code", "close"],
)
display(df_parquet_cols.head())

schema = pq.read_schema("data/clean/stock_clean.parquet")
print(schema)


,date,code,close
0,2020-01-02,000002,4497.51
1,2020-01-02,600519,8495.30
2,2020-01-02,600036,193.11
3,2020-01-02,601127,11.89
4,2020-01-02,300308,247.95


date: timestamp[ns]
code: string
name: string
industry: string
open: double
close: double
high: double
low: double
volume: double
amount: double
return: double
is_extreme: bool
year: int16
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1519


In [5]:
t0 = time.time()
pd.read_csv("data/clean/stock_clean.csv")
csv_time = time.time() - t0
csv_size = os.path.getsize("data/clean/stock_clean.csv") / 1024
print(f"CSV read time: {csv_time:.3f}s  file size: {csv_size:.1f} KB")

t0 = time.time()
pd.read_parquet("data/clean/stock_clean.parquet")
parquet_time = time.time() - t0
parquet_size = os.path.getsize("data/clean/stock_clean.parquet") / 1024
print(f"Parquet read time: {parquet_time:.3f}s  file size: {parquet_size:.1f} KB")


CSV read time: 0.028s  file size: 1827.2 KB
Parquet read time: 0.006s  file size: 777.6 KB


### CSV vs Parquet Explanation

At this project scale, the difference in read speed may be modest because the stock panel is relatively small. Parquet's advantages become clearer when the dataset grows to millions of rows, when only a few columns are needed, or when repeated analytical reads are required. CSV remains easy to inspect and exchange, but it has weaker type information and must usually parse the whole file on every read.


## 3.2 Wide and Long Transformations

The 10 stocks' closing prices are first combined into a wide table with one stock per column and `date` as the index. The same data is then converted back to long format with `pd.melt`, using the fields `date`, `code`, and `close`.


In [6]:
stock_close_wide = stock_clean_out.pivot(index="date", columns="code", values="close").sort_index()
stock_close_wide.to_csv(CLEAN_DIR / "stock_close_wide.csv", encoding="utf-8-sig")

stock_close_long = stock_close_wide.reset_index().melt(
    id_vars="date",
    var_name="code",
    value_name="close",
).dropna(subset=["close"])
stock_close_long.to_csv(CLEAN_DIR / "stock_close_long.csv", index=False, encoding="utf-8-sig")

print("Wide close table", stock_close_wide.shape)
display(stock_close_wide.head())
print("Long close table", stock_close_long.shape)
display(stock_close_long.head())


Wide close table (1540, 10)


code,000002,000858,002352,002594,300308,600036,600519,601088,601127,601398
date,,,,,,,,,,
2020-01-02,4497.51,2360.20,129.54,49.09,247.95,193.11,8495.30,31.18,11.89,10.64
2020-01-03,4427.07,2332.86,128.53,48.96,245.31,195.69,8108.58,31.13,11.90,10.67
2020-01-06,4352.48,2308.74,127.36,49.20,241.37,194.90,8104.29,31.49,11.80,10.64
2020-01-07,4387.01,2311.77,130.30,48.97,243.18,194.45,8228.64,31.56,11.93,10.71
2020-01-08,4375.96,2303.20,128.74,48.19,236.96,190.77,8180.60,30.77,12.00,10.53


Long close table (15386, 3)


,date,code,close
0,2020-01-02,000002,4497.51
1,2020-01-03,000002,4427.07
2,2020-01-06,000002,4352.48
3,2020-01-07,000002,4387.01
4,2020-01-08,000002,4375.96


### Wide vs Long Explanation

Wide tables are convenient for cross-sectional time-series operations such as correlation matrices, portfolio return calculations, and comparing multiple stock price series on the same date axis. Long tables are better for grouping, filtering, panel regression, visualization libraries, and merging with other observations by `code + date` or `code + year`.


## 3.3 Multi-table Merging

This section merges the cleaned stock daily data with index daily data using a left join on `date`. It then handles the frequency mismatch between daily stock data and monthly macro indicators by mapping each trading day to its calendar month and joining the monthly macro value to all trading days in that month.


In [7]:
index_columns = ["date", "index_code", "index_name", "purpose", "open", "close", "high", "low", "volume"]
index_frames = []

for path in sorted(INDEX_DIR.glob("index_*.csv")):
    idx = pd.read_csv(path, dtype=str)
    idx.columns = index_columns[:len(idx.columns)]
    idx["date"] = pd.to_datetime(idx["date"], errors="coerce")
    idx["index_code"] = idx["index_code"].astype(str).str.extract(r"(\d{6})", expand=False).fillna(path.stem.replace("index_", ""))
    for col in ["open", "close", "high", "low", "volume"]:
        idx[col] = pd.to_numeric(idx[col].str.replace(",", "", regex=False), errors="coerce")
    idx = idx.sort_values("date")
    idx["index_return"] = idx["close"].pct_change()
    index_frames.append(idx[["date", "index_code", "close", "index_return"]])

index_long = pd.concat(index_frames, ignore_index=True)
index_wide = index_long.pivot(index="date", columns="index_code", values=["close", "index_return"])
index_wide.columns = [f"index_{code}_{field}" for field, code in index_wide.columns]
index_wide = index_wide.rename(columns={
    "index_000300_close": "hs300_close",
    "index_000300_index_return": "hs300_return",
    "index_000001_close": "sse_close",
    "index_000001_index_return": "sse_return",
}).reset_index()

stock_daily = stock_clean_out.copy()
stock_rows_before = len(stock_daily)
index_rows = len(index_wide)
stock_index_daily = stock_daily.merge(index_wide, on="date", how="left")
stock_index_daily.to_csv(COMBINED_DIR / "stock_index_daily.csv", index=False, encoding="utf-8-sig")

merge_log = pd.DataFrame([
    {
        "merge_step": "stock daily left join index daily by date",
        "left_rows_before": stock_rows_before,
        "right_rows_before": index_rows,
        "rows_after": len(stock_index_daily),
        "row_change": len(stock_index_daily) - stock_rows_before,
    }
])

display(index_wide.head())
display(merge_log)


,date,sse_close,hs300_close,sse_return,hs300_return
0,2020-01-02,3085.198,4152.241,NaN,NaN
1,2020-01-03,3083.786,4144.965,-0.000458,-0.001752
2,2020-01-06,3083.408,4129.295,-0.000123,-0.003780
3,2020-01-07,3104.802,4160.227,0.006938,0.007491
4,2020-01-08,3066.893,4112.317,-0.012210,-0.011516


,merge_step,left_rows_before,right_rows_before,rows_after,row_change
0,stock daily left join index daily by date,15386,1540,15386,0


### Stock and Index Merge Explanation

Before the merge, each row is one stock on one trading day. After the left join, the row count should remain the same because stock observations are the left table and index values are appended by date. If some index dates are unavailable, the row count still does not change, but the index columns may contain missing values. This design keeps all stock observations for later CAPM and regression analysis.


In [8]:
macro_path = COMBINED_DIR / "combined_macro_indicators.csv"
macro_raw = pd.read_csv(macro_path, dtype=str)

# Standardize macro columns by position because this source file uses Chinese headers.
macro_columns = ["date", "indicator", "indicator_name_source", "value", "unit", "frequency", "source"]
macro_raw.columns = macro_columns[:len(macro_raw.columns)]
macro_raw["date"] = pd.to_datetime(macro_raw["date"], errors="coerce")
macro_raw["value"] = pd.to_numeric(macro_raw["value"].str.replace(",", "", regex=False), errors="coerce")
macro_raw["month"] = macro_raw["date"].dt.to_period("M").astype(str)

monthly_indicators = {
    "CPI_YOY": "cpi_yoy",
    "M2_YOY": "m2_yoy",
    "LPR_1Y": "lpr_1y",
    "INDUSTRIAL_VALUE_ADDED_YOY": "industrial_value_added_yoy",
}

macro_monthly = macro_raw[macro_raw["indicator"].isin(monthly_indicators)].copy()
macro_monthly["indicator_name"] = macro_monthly["indicator"].map(monthly_indicators)
macro_monthly = macro_monthly.sort_values("date").drop_duplicates(["month", "indicator_name"], keep="last")
macro_wide = macro_monthly.pivot(index="month", columns="indicator_name", values="value").reset_index()

stock_index_daily["month"] = pd.to_datetime(stock_index_daily["date"]).dt.to_period("M").astype(str)
rows_before_macro = len(stock_index_daily)
combined_data = stock_index_daily.merge(macro_wide, on="month", how="left")
combined_data.to_csv(COMBINED_DIR / "combined_data.csv", index=False, encoding="utf-8-sig")

macro_merge_log = pd.DataFrame([
    {
        "merge_step": "daily stock-index left join monthly macro by month",
        "left_rows_before": rows_before_macro,
        "right_rows_before": len(macro_wide),
        "rows_after": len(combined_data),
        "row_change": len(combined_data) - rows_before_macro,
    }
])

display(macro_wide.head())
display(macro_merge_log)
print("combined_data.csv", combined_data.shape)


indicator_name,month,cpi_yoy,industrial_value_added_yoy,lpr_1y,m2_yoy
0,2020-01,4.5,NaN,4.15,8.4
1,2020-02,5.4,NaN,4.05,8.8
2,2020-03,5.2,-1.1,4.05,10.1
3,2020-04,4.3,3.9,3.85,11.1
4,2020-05,3.3,4.4,3.85,11.1


,merge_step,left_rows_before,right_rows_before,rows_after,row_change
0,daily stock-index left join monthly macro by m...,15386,76,15386,0


combined_data.csv (15386, 22)


### Macro Merge Explanation

The macro series are lower-frequency monthly variables, while stock and index data are daily trading observations. To resolve the frequency mismatch, each trading day is assigned a `YYYY-MM` month key and receives the macro value for that month. The left join should preserve the number of daily stock-index rows; row changes would indicate duplicated month-indicator records or an unintended many-to-many merge. Missing macro values can still occur for months where the macro source does not publish a value or the latest month is not yet available.
